<a href="https://colab.research.google.com/github/Pmalik24/Automated_podcast_sumarization__and_highlight_generation/blob/main/speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ***DO NOT DOWNLOAD FROM GITHUB AND UNZIP AS IT PARSES CHARACTERS IN THE FILE NAMES WRONG AND THEN LATER ON TRANSCRIPT'S DONT MATCH WITH THEIR FILE PATHS, INSTEAD NORMALLY UNZIP FILE ON THE LOCAL MACHINE AND PROCEED***

In [ ]:
# import transcripts
import requests
import zipfile
import os
import shutil

# url = 'https://github.com/Pmalik24/Automated_podcast_sumarization__and_highlight_generation/raw/main/Cleaned.zip'


# # delete file if already exists
# if os.path.exists('Cleaned'):
#   shutil.rmtree('Cleaned')
# os.makedirs('Cleaned', exist_ok=True)

# # Download the file
# response = requests.get(url)
# zip_file_path = 'Cleaned.zip'

# # Save the file
# with open(zip_file_path, 'wb') as file:
#     file.write(response.content)

# # Unzip
# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#       zip_ref.extractall()

# # delete zip file
# os.remove(zip_file_path)

In [ ]:
## Importing the data
import csv
import pandas as pd

csv_file_path = 'final_df_with_transcripts.csv'

# Read the CSV file into a DataFrame using Pandas
df = pd.read_csv(csv_file_path)
df.drop('Unnamed: 0', axis=1, inplace=True)
df['Index'] = [int(i) for i in range(len(df))]

# Initialize an empty dictionary to store the data
data_dict = {}

# Iterate over each column in the DataFrame
for column in df.columns:
    # Extract values from the column and convert to a list
    values = df[column][:544].tolist()
    
    # Store the list of values in the data dictionary
    data_dict[column] = values

In [ ]:
import os

# Create a list to store transcript contents
transcript_list = []

# Loop through each transcript file path
for path in data_dict['transcription_file']:
    # Construct the full path to the transcript file
    full_path = os.path.join('Cleaned', os.path.splitext(path)[0] + '.txt')
    
    # Check if the file exists
    if os.path.exists(full_path):
        # Read the content of the transcript file and append it to the list
        with open(full_path, 'r') as file:
            transcript_list.append(file.read())
    else:
        print("The following file could not be found:", full_path)

# Assign the transcript list to the 'transcript' key in data_dict
data_dict['transcript'] = transcript_list


### **Created Df with cleaned_transcripts added to the csv**

In [ ]:
# Convert the 'transcript' values from data_dict to a Series
transcript_series = pd.Series(data_dict['transcript'])

# Add the transcript Series as a new column named 'transcript' to the DataFrame 'df'
df['transcript'] = transcript_series

# df.to_csv('final_df_with_cleaned_transcripts.csv')

### **Train Test Split Setup**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
data_df = pd.DataFrame(data_dict)
train_df, val_df = train_test_split(data_df, test_size=0.1, random_state=42)


### **Processing Audio Prep**

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch


# Load the processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# # Using the CPU for processing (or MPS/CUDA if available)
device = torch.device("cpu")  # Change "cpu" to "mps" if using Apple Silicon and the operation is supported
model.to(device)


### **DOWNLOADING AND PREPROCESSING AUDIO**

In [ ]:
import requests
from scipy.io import wavfile
import tempfile
import soundfile as sf
import librosa
import tempfile

from pydub import AudioSegment

def convert_mp3_to_wav(mp3_file_path, wav_file_path):
    audio = AudioSegment.from_mp3(mp3_file_path)
    audio.export(wav_file_path, format="wav")


def download_and_preprocess_audio(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to download audio from {url}")
    
    # Save the MP3 audio file temporarily
    with tempfile.NamedTemporaryFile(delete=False, suffix='.mp3') as tmp_mp3_file:
        tmp_mp3_file.write(response.content)

    # Convert MP3 to WAV
    tmp_wav_path = tmp_mp3_file.name.replace('.mp3', '.wav')
    convert_mp3_to_wav(tmp_mp3_file.name, tmp_wav_path)

    # Load and resample the WAV audio file using librosa or soundfile
    audio, sample_rate = librosa.load(tmp_wav_path, sr=16000)
    
    return audio

### **Audio To Text Fn**

In [ ]:

def audio_to_text(model, processor, audio):
    # The processor replaces the tokenizer for both feature extraction and decoding
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding="longest").input_values
    inputs = inputs.to(device)
    
    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription

### **Applying to Df**

In [ ]:
from tqdm.notebook import tqdm
import traceback

def process_audio_file(url, model, processor, device):
    try:
        audio = download_and_preprocess_audio(url)
        text = audio_to_text(model, processor, audio)
        return text
    except Exception as e:
        print(f"Error processing {url}: {e}")
        traceback.print_exc()
        return None

In [ ]:
def process_dataset(df, model, processor, device):
    # Initialize an empty list to store the converted texts
    converted_texts = []
    
    # Iterate over the DataFrame with a tqdm progress bar
    for url in tqdm(df['download_url'], desc="Processing audio files"):
        converted_text = process_audio_file(url, model, processor, device)
        converted_texts.append(converted_text)
    
    # Add the converted texts to the DataFrame as a new column
    df['converted_text'] = converted_texts




In [ ]:
# Example of processing the validation dataset
process_dataset(train_df, model, processor, device)